# YOLOv8 для обнаружения объектов

**Модель:** YOLOv8 (You Only Look Once version 8)

**Особенности архитектуры:**
- Одноэтапный детектор - прямое предсказание из сетки
- Anchor-free подход (не использует якорные боксы)
- CSPDarknet backbone для извлечения признаков
- PANet для объединения мультимасштабных признаков
- Децентрализованные головы для bbox и классификации

**Преимущества:**
- Очень быстрая скорость inference (real-time)
- Отличный баланс точности и скорости
- Простота использования через ultralytics
- Меньше параметров, чем у Faster R-CNN

**Недостатки:**
- Может быть менее точным на мелких объектах
- Требует правильной настройки гиперпараметров

**Главное отличие от Faster R-CNN:**
- Faster R-CNN: двухэтапный (RPN → классификация)
- YOLO: одноэтапный (напрямую предсказывает bbox и классы за один проход)

In [ ]:
# Установка необходимых библиотек
# YOLO использует ultralytics вместо чистого PyTorch
!pip install ultralytics torch torchvision matplotlib pillow numpy -q

In [ ]:
# Подключение Google Drive
from google.colab import drive
drive.mount('/content/drive')

print('Google Drive подключен успешно!')

In [ ]:
from ultralytics import YOLO
import torch
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import yaml

print(f'PyTorch версия: {torch.__version__}')
print(f'CUDA доступна: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA устройство: {torch.cuda.get_device_name(0)}')

In [ ]:
# Конфигурация путей к данным
DATA_ROOT = '/content/drive/MyDrive/Датасет'  # Измените на свой путь

# Гиперпараметры
NUM_CLASSES = 5  # YOLO считает только объектные классы (без фона)
BATCH_SIZE = 16  # YOLO эффективнее, можно больший batch
NUM_EPOCHS = 10
IMG_SIZE = 640  # Стандартный размер для YOLO
DEVICE = '0' if torch.cuda.is_available() else 'cpu'  # YOLO использует строку для device

print(f'Используемое устройство: {DEVICE}')

In [ ]:
# ============================================================
# СОЗДАНИЕ КОНФИГУРАЦИОННОГО ФАЙЛА ДЛЯ YOLO
# ============================================================
#
# YOLO требует YAML файл с описанием датасета
# В отличие от Faster R-CNN, где мы использовали Dataset class,
# YOLO использует свой встроенный загрузчик данных
#
# YOLO ожидает структуру:
# - path: корневая директория
# - train/val/test: относительные пути к изображениям
# - names: словарь с именами классов
# ============================================================

# Создание YAML конфигурации для датасета
data_yaml = {
    'path': DATA_ROOT,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': NUM_CLASSES,  # number of classes
    'names': {i: f'class_{i}' for i in range(NUM_CLASSES)}  # Замените на ваши классы
}

# Сохранение конфигурации
yaml_path = '/content/dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)

print('YAML конфигурация создана:')
print(yaml.dump(data_yaml, default_flow_style=False))

In [ ]:
# ============================================================
# СОЗДАНИЕ МОДЕЛИ YOLOV8
# ============================================================
#
# YOLOv8 имеет несколько размеров:
# - yolov8n.pt (nano) - самая быстрая, наименее точная
# - yolov8s.pt (small) - баланс скорости и точности
# - yolov8m.pt (medium) - хорошая точность
# - yolov8l.pt (large) - высокая точность
# - yolov8x.pt (xlarge) - максимальная точность
#
# Архитектура YOLOv8:
# 1. Backbone (CSPDarknet) - извлечение признаков
# 2. Neck (PANet) - объединение мультимасштабных признаков
# 3. Head - децентрализованные головы для bbox и классов
#
# Отличие от Faster R-CNN:
# - Faster R-CNN: RPN генерирует предложения → классификация
# - YOLOv8: напрямую предсказывает bbox и классы на выходе сети
# - YOLOv8 использует anchor-free подход (без якорей)
# - YOLOv8 гораздо быстрее (одноэтапный детектор)
# ============================================================

# Загрузка предобученной модели YOLOv8
# Используем yolov8s для баланса скорости и точности
model = YOLO('yolov8s.pt')  # автоматически загрузит веса с интернета

print('Модель YOLOv8s загружена')
print(f'Количество классов: {NUM_CLASSES}')

In [ ]:
# ============================================================
# ОБУЧЕНИЕ YOLOV8
# ============================================================
#
# YOLO использует встроенный метод .train() который:
# - Автоматически загружает данные из YAML
# - Применяет аугментацию данных
# - Использует оптимизатор AdamW или SGD
# - Сохраняет лучшие веса автоматически
#
# Отличие от Faster R-CNN:
# - Faster R-CNN: нужно писать свои циклы train/val
# - YOLO: все встроено в библиотеку ultralytics
# ============================================================

# Обучение модели
print('Начало обучения YOLOv8...')
results = model.train(
    data=yaml_path,
    epochs=NUM_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    patience=50,  # early stopping
    save=True,
    plots=True,  # автоматические графики
    verbose=True
)

print('Обучение завершено!')
print(f'Лучшие веса сохранены в: {results.save_dir}')

In [ ]:
# Визуализация графиков обучения
# YOLO автоматически создает графики, но мы можем создать свои

# Загрузка результатов обучения
results_csv = f'{results.save_dir}/results.csv'
if os.path.exists(results_csv):
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()  # Удалить пробелы
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # График лоссов
    axes[0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', marker='o')
    axes[0].plot(df['epoch'], df['train/cls_loss'], label='Train Class Loss', marker='s')
    axes[0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', marker='^')
    axes[0].plot(df['epoch'], df['val/cls_loss'], label='Val Class Loss', marker='d')
    axes[0].set_xlabel('Эпоха', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('График Loss YOLOv8', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    # График метрик
    axes[1].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', marker='o')
    axes[1].plot(df['epoch'], df['metrics/recall(B)'], label='Recall', marker='s')
    axes[1].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50', marker='^')
    axes[1].set_xlabel('Эпоха', fontsize=12)
    axes[1].set_ylabel('Значение', fontsize=12)
    axes[1].set_title('График метрик YOLOv8', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 1])
    
    # Подсказки
    axes[0].text(0.02, 0.98, 
        '💡 Интерпретация:\n'
        '✅ Loss должен снижаться\n'
        '⚠️ Если val loss растет - переобучение\n'
        '📊 Меньше = Лучше',
        transform=axes[0].transAxes, fontsize=10,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    axes[1].text(0.02, 0.02, 
        '💡 Интерпретация:\n'
        '✅ Метрики должны расти\n'
        'mAP50 - главная метрика точности\n'
        '📊 Больше = Лучше',
        transform=axes[1].transAxes, fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    print(f'Финальный mAP50: {df["metrics/mAP50(B)"].iloc[-1]:.4f}')
else:
    print('Файл results.csv не найден')

In [ ]:
# ============================================================
# ВАЛИДАЦИЯ YOLOV8
# ============================================================
#
# YOLO имеет встроенный метод .val() который:
# - Вычисляет метрики (Precision, Recall, mAP50, mAP50-95)
# - Использует свои оптимизированные функции IoU
# - Автоматически применяет NMS (Non-Maximum Suppression)
#
# Отличие от Faster R-CNN:
# - Faster R-CNN: вручную считаем метрики через IoU
# - YOLO: встроенная валидация с оптимизациями
# ============================================================

# Валидация на тестовом наборе
print('Валидация на тестовом наборе...')
test_results = model.val(
    data=yaml_path,
    split='test',  # используем тестовый набор
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True
)

# Вывод метрик
print('\n' + '='*50)
print('МЕТРИКИ НА ТЕСТОВОМ НАБОРЕ (YOLOv8)')
print('='*50)
print(f'Precision: {test_results.results_dict["metrics/precision(B)"]:.4f}')
print(f'Recall: {test_results.results_dict["metrics/recall(B)"]:.4f}')
print(f'mAP50: {test_results.results_dict["metrics/mAP50(B)"]:.4f}')
print(f'mAP50-95: {test_results.results_dict["metrics/mAP50-95(B)"]:.4f}')
print('='*50)

print('\n💡 Интерпретация метрик:')
print('Precision - какой % предсказаний правильный (больше = лучше)')
print('Recall - какой % объектов найден (больше = лучше)')
print('mAP50 - средняя точность при IoU=0.5 (главная метрика)')
print('mAP50-95 - средняя точность при IoU от 0.5 до 0.95')

In [ ]:
# Визуализация метрик
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

metrics_names = ['Precision', 'Recall', 'mAP50', 'mAP50-95']
metrics_values = [
    test_results.results_dict['metrics/precision(B)'],
    test_results.results_dict['metrics/recall(B)'],
    test_results.results_dict['metrics/mAP50(B)'],
    test_results.results_dict['metrics/mAP50-95(B)']
]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

bars = ax.bar(metrics_names, metrics_values, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Значение', fontsize=12)
ax.set_title('Метрики качества YOLOv8 на тестовом наборе', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

# Добавление значений на столбцы
for bar, value in zip(bars, metrics_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.3f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

# Подсказка
ax.text(0.5, 0.95, '📊 Больше = Лучше (максимум = 1.0)',
        transform=ax.transAxes, fontsize=11,
        ha='center', va='top',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# INFERENCE (ПРЕДСКАЗАНИЕ) YOLOV8
# ============================================================
#
# YOLO использует метод .predict() для inference:
# - Автоматически применяет NMS (подавление немаксимальных значений)
# - Возвращает Results объект с bbox, классами, confidence
# - Может работать с изображениями, видео, стримами
#
# Отличие от Faster R-CNN:
# - Faster R-CNN: model(images) возвращает словарь
# - YOLO: model.predict() возвращает Results объект
# - YOLO имеет встроенную визуализацию через .plot()
# ============================================================

# Визуализация предсказаний на тестовых изображениях
test_img_dir = os.path.join(DATA_ROOT, 'test/images')
test_images = [os.path.join(test_img_dir, f) for f in os.listdir(test_img_dir) if f.endswith('.png')]

# Выбираем 9 случайных изображений
selected_images = np.random.choice(test_images, min(9, len(test_images)), replace=False)

fig, axes = plt.subplots(3, 3, figsize=(20, 20))
axes = axes.flatten()

for idx, (img_path, ax) in enumerate(zip(selected_images, axes)):
    # Предсказание
    results = model.predict(img_path, conf=0.1, device=DEVICE, verbose=False)
    
    # Загрузка изображения
    img = Image.open(img_path)
    ax.imshow(img)
    ax.axis('off')
    
    # Загрузка Ground Truth
    label_path = img_path.replace('images', 'labels').replace('.png', '.txt')
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f.readlines():
                class_id, x_center, y_center, width, height = map(float, line.strip().split())
                img_width, img_height = img.size
                x_min = (x_center - width / 2) * img_width
                y_min = (y_center - height / 2) * img_height
                box_width = width * img_width
                box_height = height * img_height
                
                rect = patches.Rectangle((x_min, y_min), box_width, box_height,
                                         linewidth=2, edgecolor='green',
                                         facecolor='none', label='GT')
                ax.add_patch(rect)
                ax.text(x_min, y_min-5, f'GT: {int(class_id)}',
                       color='green', fontsize=10, weight='bold',
                       bbox=dict(facecolor='white', alpha=0.7))
    
    # Отрисовка предсказаний YOLO
    for result in results:
        boxes = result.boxes.cpu().numpy()
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            conf = box.conf[0]
            cls = int(box.cls[0])
            
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                     linewidth=2, edgecolor='red',
                                     facecolor='none', linestyle='--')
            ax.add_patch(rect)
            ax.text(x2, y2+5, f'Pred: {cls} ({conf:.2f})',
                   color='red', fontsize=10, weight='bold',
                   bbox=dict(facecolor='white', alpha=0.7))
    
    ax.set_title(f'Изображение {idx}', fontsize=12)

plt.suptitle('Результаты обнаружения YOLOv8\n(Зеленый=Ground Truth, Красный=Предсказание)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('Визуализация завершена!')

## Заключение

**YOLOv8** - это современный одноэтапный детектор объектов:

**Когда использовать:**
- Для real-time приложений (видеонаблюдение, автономные машины)
- Когда нужен баланс между скоростью и точностью
- Когда важна простота использования
- Для мобильных и edge устройств (используйте yolov8n)

**Когда НЕ использовать:**
- Когда критична максимальная точность (используйте Faster R-CNN)
- Для очень мелких объектов (может быть менее точным)

**Сравнение с Faster R-CNN:**
- ⚡ Скорость: YOLO >> Faster R-CNN
- 🎯 Точность: Faster R-CNN ≥ YOLO (зависит от задачи)
- 💾 Память: YOLO < Faster R-CNN
- 📱 Простота: YOLO >> Faster R-CNN (ultralytics API)